In [ ]:
!pip install python-dotenv wandb unsloth torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 5.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.3/299.3 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.1/117.1 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.2/821.2 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 139.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7

In [ ]:
import os
import wandb


from google.colab import userdata

def setup_wandb(project_name: str, run_name: str):
    # Set up your API KEY
    try:
        api_key = userdata.get("WANDB_API_KEY")
        wandb.login(key=api_key)
        print("Successfully logged into WandB.")
    except KeyError:
        raise EnvironmentError("WANDB_API_KEY is not set in the environment variables.")
    except Exception as e:
        print(f"Error logging into WandB: {e}")

    # Optional: Log models
    os.environ["WANDB_LOG_MODEL"] = "checkpoint"

    os.environ["WANDB_WATCH"] = "all"
    os.environ["WANDB_SILENT"] = "true"

    # Initialize the WandB run
    try:
        wandb.init(project=project_name, name=run_name)
        print(f"WandB run initialized: Project - {project_name}, Run - {run_name}")
    except Exception as e:
        print(f"Error initializing WandB run: {e}")


setup_wandb(project_name="custom_siri", run_name="second_iteration")

Error logging into WandB: Secret WANDB_API_KEY does not exist.


<IPython.core.display.Javascript object>

wandb: Paste an API key from your profile and hit enter:

 ··········
WandB run initialized: Project - custom_siri, Run - second_iteration


In [ ]:
from huggingface_hub import login
from getpass import getpass

hf_token = getpass("Paste your Hugging Face token: ")
login(hf_token)

Paste your Hugging Face token: ··········


LOAD BASE MODEL

In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048     # Unsloth auto supports RoPE Scaling internally!
dtype = None              # None for auto detection
load_in_4bit = True      # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Meta-Llama-3.1-8B-Instruct", #(or "unsloth/gemma-2b-it")
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.7.6: Fast Gemma patching. Transformers: 4.53.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,   # LoRA rank - suggested values: 8, 16, 32, 64, 128
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,   # Supports any, but = 0 is optimized
    bias="none",      # Supports any, but = "none" is optimized
    use_gradient_checkpointing="unsloth",  # Ideal for long context tuning
    random_state=3407,
    use_rslora=False,   # Disable rank-sensitive LoRA for simpler tasks
    loftq_config=None   # No LoftQ, for standard fine-tuning
)

Unsloth 2025.7.6 patched 18 layers with 18 QKV layers, 18 O layers and 18 MLP layers.


In [ ]:
from datasets import load_dataset

# Loading the dataset
dataset = load_dataset("valex95/siri-function-calling-v4", split="train", token=hf_token)


print(f"Using a sample size of {len(dataset)} for fine-tuning.")

README.md:   0%|          | 0.00/373 [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/43.1k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4500 [00:00<?, ? examples/s]

Using a sample size of 4500 for fine-tuning.


In [ ]:
!pip install protobuf==3.20.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 12.7 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
ydf 0.13.0 requires protobuf<7.0.0,>=5.29.1, but you have protobuf 3.20.3 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 3.20.3 which is incompatible.


In [ ]:
from unsloth.chat_templates import get_chat_template

# Initialize the tokenizer with the chat template and mapping
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3", #(or "gemma")
    mapping = {"role" : "from", "content" : "value", "user" : "human", "assistant" : "gpt"}, # ShareGPT style
    map_eos_token = True,        # Maps <|im_end|> to <|eot_id|> instead
)

def formatting_prompts_func(examples):
    convos = []

    # Iterate through each item in the batch (examples are structured as lists of values)
    for query, tools, answers in zip(examples['query'], examples['tools'], examples['answers']):
        tool_user = {
            "content": f"You are a helpful assistant with access to the following tools or function calls. Your task is to produce a sequence of tools or function calls necessary to generate response to the user utterance. Use the following tools or function calls as required:\n{tools}",
            "role": "system"
        }
        ques_user = {
            "content": f"{query}",
            "role": "user"
        }
        assistant = {
            "content": f"{answers}",
            "role": "assistant"
        }
        convos.append([tool_user, ques_user, assistant])

    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts}


# Apply the formatting on dataset
dataset = dataset.map(formatting_prompts_func, batched = True,)

Unsloth: Will map <end_of_turn> to EOS = <eos>.


Map:   0%|          | 0/4500 [00:00<?, ? examples/s]

In [ ]:
from transformers import TrainingArguments

args = TrainingArguments(
        per_device_train_batch_size = 8,  # Controls the batch size per device
        gradient_accumulation_steps = 2,  # Accumulates gradients to simulate a larger batch
        warmup_steps = 5,
        learning_rate = 2e-4,             # Sets the learning rate for optimization
        num_train_epochs = 3,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        optim = "adamw_8bit",
        weight_decay = 0.01,              # Regularization term for preventing overfitting
        lr_scheduler_type = "linear",     # Chooses a linear learning rate decay
        seed = 3407,
        output_dir = "outputs",
        report_to = "wandb",              # Enables Weights & Biases (W&B) logging
        logging_steps = 1,                # Sets frequency of logging to W&B
        logging_strategy = "steps",       # Logs metrics at each specified step
        save_strategy = "no",
        load_best_model_at_end = True,    # Loads the best model at the end
        save_only_model = False           # Saves entire model, not only weights
    )

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,        # Can make training 5x faster for short sequences.
    args = args
)

Unsloth: Tokenizing ["text"]:   0%|          | 0/4500 [00:00<?, ? examples/s]

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.161 GB.
2.871 GB of memory reserved.


In [ ]:
from unsloth import unsloth_train

trainer_stats = unsloth_train(trainer)
print(trainer_stats)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,500 | Num Epochs = 3 | Total steps = 846
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 19,611,648 of 2,525,784,064 (0.78% trained)
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: cosmin-claudiu (alex-vesa-cube-digital) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,2.732700
2,2.819500
3,2.744600
4,2.860900
5,2.533300
6,2.459700
7,2.204500
8,2.018300
9,1.949600
10,1.755000


TrainOutput(global_step=846, training_loss=0.18621256116152374, metrics={'train_runtime': 1000.633, 'train_samples_per_second': 13.491, 'train_steps_per_second': 0.845, 'total_flos': 2.786418772426752e+16, 'train_loss': 0.18621256116152374})


In [ ]:
wandb.finish()

NameError: name 'wandb' is not defined

In [ ]:
model.push_to_hub("YourUser/YourModel", token = hf_token)
tokenizer.push_to_hub("YourUser/YourTokenzier", token = hf_token)

README.md:   0%|          | 0.00/576 [00:00<?, ?B/s]

Uploading...:   0%|          | 0.00/78.5M [00:00<?, ?B/s]

Saved model to https://huggingface.co/CosminMihai02/gemma_2b_custom_siri_model_v1


Uploading...:   0%|          | 0.00/34.4M [00:00<?, ?B/s]

In [ ]:
!pip install unsloth
from unsloth import FastLanguageModel

max_seq_length = 2048
dtype = None
load_in_4bit = True
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="YourUser/YourModel",  # Trained model either locally or from huggingface
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.3/47.3 kB 3.0 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.3/299.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.2/376.2 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.0/166.0 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.1/117.1 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.2/821.2 MB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.7/155.7 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 133.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7

model.safetensors:   0%|          | 0.00/2.07G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/154 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/78.5M [00:00<?, ?B/s]

Unsloth 2025.7.6 patched 18 layers with 18 QKV layers, 18 O layers and 18 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): GemmaForCausalLM(
      (model): GemmaModel(
        (embed_tokens): Embedding(256000, 2048, padding_idx=0)
        (layers): ModuleList(
          (0-17): 18 x GemmaDecoderLayer(
            (self_attn): GemmaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Lin

In [ ]:
!pip install pycaw comtypes
import os
import shutil
import subprocess
import webbrowser
import json
import psutil
import time
import platform

# --- File operations ---

def copy_file(source: str, destination: str) -> str:
    """Copies a file from source to destination."""
    try:
        shutil.copy(source, destination)
        return f"Copied {source} to {destination}"
    except Exception as e:
        return f"Failed to copy file: {e}"

def move_file(source: str, destination: str) -> str:
    """Moves a file from source to destination."""
    try:
        shutil.move(source, destination)
        return f"Moved {source} to {destination}"
    except Exception as e:
        return f"Failed to move file: {e}"

def delete_file(path: str) -> str:
    """Deletes the file at the given path."""
    try:
        os.remove(path)
        return f"Deleted file {path}"
    except Exception as e:
        return f"Failed to delete file: {e}"

def create_folder(path: str) -> str:
    """Creates a folder at the given path."""
    try:
        os.makedirs(path, exist_ok=True)
        return f"Created folder at {path}"
    except Exception as e:
        return f"Failed to create folder: {e}"

# --- System commands ---

def open_application(app_name: str) -> str:
    """Opens an application by name."""
    try:
        subprocess.Popen(["open", "-a", app_name])
        return f"Opened {app_name}"
    except Exception as e:
        return f"Failed to open {app_name}: {e}"

def close_application(app_name: str) -> str:
    """Closes an application by name."""
    try:
        for proc in psutil.process_iter(['name']):
            if app_name.lower() in proc.info['name'].lower():
                proc.kill()
        return f"Closed {app_name}"
    except Exception as e:
        return f"Failed to close {app_name}: {e}"

def take_screenshot(destination: str) -> str:
    """Takes a screenshot and saves it."""
    try:
        print('screenshot')
        return f"Screenshot saved to {destination}"
    except Exception as e:
        return f"Failed to take screenshot: {e}"

def lock_screen() -> str:
    """Locks the laptop screen."""
    try:
        subprocess.run(["pmset", "displaysleepnow"])
        return "Screen locked"
    except Exception as e:
        return f"Failed to lock screen: {e}"

def get_battery_status() -> dict:
    """Returns the battery level and charging status."""
    try:
        battery = psutil.sensors_battery()
        return {
            "percent": battery.percent,
            "charging": battery.power_plugged
        }
    except Exception as e:
        return {"error": str(e)}

# --- Browser / Internet ---

def open_url(url: str) -> str:
    """Opens a URL in the default web browser."""
    try:
        webbrowser.open(url)
        return f"Opened {url}"
    except Exception as e:
        return f"Failed to open URL: {e}"

def search_google(query: str) -> str:
    """Searches Google for the given query."""
    try:
        url = f"https://www.google.com/search?q={query.replace(' ', '+')}"
        webbrowser.open(url)
        return f"Searched for: {query}"
    except Exception as e:
        return f"Failed to search: {e}"

# --- Media ---

def play_music(track_name: str) -> str:
    """Pretends to play a music track."""
    return f"Now playing: {track_name}"

def pause_music() -> str:
    """Pretends to pause music playback."""
    return "Music paused"


def set_volume(level: int) -> str:
    if platform.system() == "Darwin":
        try:
            subprocess.run(["osascript", "-e", f"set volume output volume {level}"])
            return f"Volume set to {level}%"
        except Exception as e:
            return f"Failed to set volume: {e}"
    elif platform.system() == "Windows":
        try:
            from ctypes import cast, POINTER
            from comtypes import CLSCTX_ALL
            from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume

            level = max(0, min(level, 100))
            devices = AudioUtilities.GetSpeakers()
            interface = devices.Activate(IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
            volume = cast(interface, POINTER(IAudioEndpointVolume))
            volume.SetMasterVolumeLevelScalar(level / 100.0, None)
            return f"Volume set to {level}%"
        except Exception as e:
            return f"Failed to set volume: {e}"
    else:
        return "Volume control not supported on this OS."

# --- Notes ---

def create_note(title: str, content: str) -> str:
    """Creates a mock note."""
    return f"Note created: {title} - {content}"

functions = [
    {
        "name": "copy_file",
        "description": "Copies a file from source to destination.",
        "parameters": {
            "type": "object",
            "properties": {
                "source": {"type": "string", "description": "Path of the source file"},
                "destination": {"type": "string", "description": "Path of the destination"}
            },
            "required": ["source", "destination"]
        }
    },
    {
        "name": "move_file",
        "description": "Moves a file from source to destination.",
        "parameters": {
            "type": "object",
            "properties": {
                "source": {"type": "string"},
                "destination": {"type": "string"}
            },
            "required": ["source", "destination"]
        }
    },
    {
        "name": "delete_file",
        "description": "Deletes a file.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"}
            },
            "required": ["path"]
        }
    },
    {
        "name": "create_folder",
        "description": "Creates a folder at the given path.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string"}
            },
            "required": ["path"]
        }
    },
    {
        "name": "open_application",
        "description": "Opens an application by name.",
        "parameters": {
            "type": "object",
            "properties": {
                "app_name": {"type": "string"}
            },
            "required": ["app_name"]
        }
    },
    {
        "name": "close_application",
        "description": "Closes an application by name.",
        "parameters": {
            "type": "object",
            "properties": {
                "app_name": {"type": "string"}
            },
            "required": ["app_name"]
        }
    },
    {
        "name": "take_screenshot",
        "description": "Takes a screenshot and saves it.",
        "parameters": {
            "type": "object",
            "properties": {
                "destination": {"type": "string"}
            },
            "required": ["destination"]
        }
    },
    {
        "name": "lock_screen",
        "description": "Locks the screen.",
        "parameters": {
            "type": "object",
            "properties": {}
        }
    },
    {
        "name": "get_battery_status",
        "description": "Returns battery percent and charging status.",
        "parameters": {
            "type": "object",
            "properties": {}
        }
    },
    {
        "name": "open_url",
        "description": "Opens a URL in the browser.",
        "parameters": {
            "type": "object",
            "properties": {
                "url": {"type": "string"}
            },
            "required": ["url"]
        }
    },
    {
        "name": "search_google",
        "description": "Searches Google.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"}
            },
            "required": ["query"]
        }
    },
    {
        "name": "play_music",
        "description": "Plays a music track (mock).",
        "parameters": {
            "type": "object",
            "properties": {
                "track_name": {"type": "string"}
            },
            "required": ["track_name"]
        }
    },
    {
        "name": "pause_music",
        "description": "Pauses playback (mock).",
        "parameters": {
            "type": "object",
            "properties": {}
        }
    },
    {
        "name": "set_volume",
        "description": "Sets the system volume (0–100).",
        "parameters": {
            "type": "object",
            "properties": {
                "level": {"type": "integer"}
            },
            "required": ["level"]
        }
    },
    {
        "name": "create_note",
        "description": "Creates a simple note.",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {"type": "string"},
                "content": {"type": "string"}
            },
            "required": ["title", "content"]
        }
    }
]




   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.4/246.4 kB 10.0 MB/s eta 0:00:00


In [ ]:
available_function_calls = {
    "copy_file": copy_file,
    "move_file": move_file,
    "delete_file": delete_file,
    "create_folder": create_folder,
    "open_application": open_application,
    "close_application": close_application,
    "take_screenshot": take_screenshot,
    "lock_screen": lock_screen,
    "get_battery_status": get_battery_status,
    "open_url": open_url,
    "search_google": search_google,
    "play_music": play_music,
    "pause_music": pause_music,
    "set_volume": set_volume,
    "create_note": create_note
}


In [ ]:
import re
import json

def extract_json_arrays(text):
    # Find all JSON array-like blocks
    array_pattern = r"\[\s*{.*?}\s*]"  # non-greedy match
    matches = re.findall(array_pattern, text, re.DOTALL)

    # Convert each match into an actual list
    parsed = []
    for match in matches:
        try:
            parsed.extend(json.loads(match))
        except json.JSONDecodeError as e:
            print("Error decoding:", match, e)
    return parsed


Exception: Expecting value: line 1 column 11 (char 10)

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
model.push_to_hub_gguf(
    repo_id="YourUser/YourOllama",
    tokenizer=tokenizer,
    quantization_method=["q4_k_m", "q5_k_m", "q8_0"],
    token=hf_token
)

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 33.56 out of 52.96 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 18/18 [00:00<00:00, 36.80it/s]


Unsloth: Saving tokenizer... Done.
Done.
==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q4_k_m', 'q5_k_m', 'q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: [1] Converting model at CosminMihai02/gemma2b_ollama_v1 into bf16 GGUF format.
The output location will be /content/CosminMihai02/gemma2b_ollama_v1/unsloth.BF16.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: gemma2b_ollama_v1
INFO:hf-to-gguf:Model architecture: GemmaForCausalLM
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: loading model part '

RuntimeError: Unsloth: Quantization failed for /content/CosminMihai02/gemma2b_ollama_v1/unsloth.BF16.gguf
You might have to compile llama.cpp yourself, then run this again.
You do not need to close this Python program. Run the following commands in a new terminal:
You must run this in the same folder as you're saving your model.
git clone --recursive https://github.com/ggerganov/llama.cpp
cd llama.cpp && make clean && make all -j
Once that's done, redo the quantization.

In [ ]:
from tqdm import tqdm

dataset = load_dataset("valex95/siri-function-calling", split="train", token=hf_token)

def evaluate_model_on_dataset(dataset, model, tokenizer, max_samples=900):
    total = 0
    correct_calls = 0
    valid_json = 0
    wrong_structure = 0

    results = []

    for example in tqdm(dataset.select(range(min(max_samples, len(dataset))))):
        query = example["query"]
        expected_calls = example["answers"] if isinstance(example["answers"], list) else json.loads(example["answers"])
        tools = example["tools"] if isinstance(example["tools"], list) else json.loads(example["tools"])

        # Prompt model with tools
        chat = [
            {"role": "system", "content": (
                "You are a function-calling assistant that only replies with JSON.\n"
                f"Available functions:\n{json.dumps(tools, indent=2)}"
            )},
            {"role": "user", "content": query}
        ]

        # Generate model response
        inputs = tokenizer.apply_chat_template(chat, tokenize=True, add_generation_prompt=True, return_tensors="pt").to(model.device)
        output = model.generate(inputs, max_new_tokens=512)

        try:
            response = tokenizer.batch_decode(output)[0]
            generated_calls = extract_json_arrays(response)
            valid_json += 1
        except Exception:
            generated_calls = []
            wrong_structure += 1

        # Compare only first function call (for now)
        expected = expected_calls[0] if expected_calls else None
        generated = generated_calls[1] if generated_calls else None

        match = (
            [generated.get("name") if generated is not None else 0] == [expected.get("name") if expected is not None else 0]
            and [generated.get("arguments") if generated is not None else 0] == [expected.get("arguments") if expected is not None else 0]
        )

        if match:
            correct_calls += 1

        total += 1
        results.append({
            "query": query,
            "expected": expected,
            "generated": generated,
            "match": match
        })

    # Print summary
    print("\n📊 Evaluation Summary:")
    print(f"Total Samples Evaluated: {total}")
    print(f"✅ Correct Function Calls: {correct_calls} ({correct_calls/total:.2%})")
    print(f"🧠 Valid JSON Outputs: {valid_json} ({valid_json/total:.2%})")
    print(f"❌ Invalid JSON/Structure: {wrong_structure} ({wrong_structure/total:.2%})")

    return results

results = evaluate_model_on_dataset(dataset, model, tokenizer)

100%|██████████| 10/10 [00:46<00:00,  4.60s/it]


📊 Evaluation Summary:
Total Samples Evaluated: 10
✅ Correct Function Calls: 7 (70.00%)
🧠 Valid JSON Outputs: 7 (70.00%)
❌ Invalid JSON/Structure: 3 (30.00%)
